# YOLOv8-Seg Training — MVTec 3D-AD (Bagel)

This notebook trains a **YOLOv8 segmentation model** (`yolov8n-seg`) on the MVTec 3D-AD bagel dataset.

> **Note :** `yolov8-seg` is the current Ultralytics standard for instance segmentation.
> The model detects and segments defect regions directly from RGB images.

### Pipeline
```
MVTec RGB images + GT masks
        ↓
Convert masks → YOLO polygon labels
        ↓
Train YOLOv8n-seg
        ↓
Evaluate: mAP50, mAP50-95, mask IoU
```

### Classes
| ID | Name | Mask value |
|---|---|---|
| 0 | crack | 254 |
| 1 | hole | 253 |
| 2 | contamination | 255 |

## 1. Setup & Installation

In [ ]:
# Install dependencies
!pip install ultralytics opencv-python-headless tqdm pyyaml --quiet

In [ ]:
import os
import sys
import shutil
import random
import yaml
import json
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from tqdm import tqdm
from PIL import Image

# Root of the project
ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(ROOT))

# Paths
DATA_RAW   = ROOT / 'data' / 'raw' / 'bagel'
YOLO_DIR   = ROOT / 'data' / 'yolo_bagel'

print('Project root :', ROOT)
print('Raw data     :', DATA_RAW)
print('YOLO dataset :', YOLO_DIR)

## 2. Data Overview

In [ ]:
# ── Dataset statistics ────────────────────────────────────────────────────────
DEFECT_CLASSES = {
    'crack':         {'id': 0, 'color': (255,  80,  80), 'mask_val': 254},
    'hole':          {'id': 1, 'color': ( 80, 180, 255), 'mask_val': 253},
    'contamination': {'id': 2, 'color': ( 80, 220,  80), 'mask_val': 255},
    'combined':      {'id': None, 'color': (255, 200,  0), 'mask_val': None},
    'good':          {'id': None, 'color': (200, 200, 200), 'mask_val':   0},
}

print("{:<18} {:>8} {:>15} {:>16}".format('Defect', 'Samples', 'Mask values', 'Avg defect area'))
print('-' * 62)
for defect, info in DEFECT_CLASSES.items():
    split_dir = DATA_RAW / 'test' / defect
    rgb_files = sorted((split_dir / 'rgb').glob('*.png'))
    gt_dir    = split_dir / 'gt'
    n = len(rgb_files)
    if gt_dir.exists() and defect != 'good':
        areas = []
        for gf in sorted(gt_dir.glob('*.png')):
            m = np.array(Image.open(gf))
            areas.append(100 * (m > 0).mean())
        avg_area    = "{:.2f}%".format(np.mean(areas))
        unique_vals = str(np.unique(np.array(Image.open(sorted(gt_dir.glob('*.png'))[0]))))
    else:
        avg_area    = 'N/A'
        unique_vals = '[0]'
    print("{:<18} {:>8} {:>15} {:>16}".format(defect, n, unique_vals, avg_area))

In [ ]:
# ── Visualise one sample per defect type ─────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
defects   = ['crack', 'hole', 'contamination', 'combined']

for col, defect in enumerate(defects):
    rgb_dir = DATA_RAW / 'test' / defect / 'rgb'
    gt_dir  = DATA_RAW / 'test' / defect / 'gt'
    rgb_file = sorted(rgb_dir.glob('*.png'))[0]
    gt_file  = sorted(gt_dir.glob('*.png'))[0]

    rgb  = np.array(Image.open(rgb_file))
    mask = np.array(Image.open(gt_file))

    # RGB with overlay
    overlay = rgb.copy()
    overlay[mask > 0] = (overlay[mask > 0] * 0.5 + np.array([255, 0, 0]) * 0.5).astype(np.uint8)
    axes[0, col].imshow(overlay)
    axes[0, col].set_title(f'{defect.upper()} — RGB + overlay', fontsize=11)
    axes[0, col].axis('off')

    # GT mask
    axes[1, col].imshow(mask, cmap='hot')
    axes[1, col].set_title(f'GT mask (unique: {np.unique(mask)})', fontsize=11)
    axes[1, col].axis('off')

plt.suptitle('MVTec 3D-AD — Bagel Dataset Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(ROOT / 'experiments' / 'results' / 'dataset_overview.png'), dpi=120)
plt.show()
print('Saved → experiments/results/dataset_overview.png')

## 3. Convert MVTec → YOLO Format

YOLO segmentation label format:
```
<class_id>  <x1> <y1>  <x2> <y2>  ...  <xn> <yn>
```
All coordinates are normalised to `[0, 1]`.

In [ ]:
def mask_to_yolo_polygons(mask: np.ndarray, class_id: int,
                          min_area: int = 30) -> list:
    """
    Convert a binary (or value-filtered) mask to YOLO polygon label lines.

    Args:
        mask      : (H, W) uint8 — non-zero pixels = defect region
        class_id  : YOLO class index
        min_area  : ignore contours smaller than this (pixels²)
    Returns:
        list of strings, one per contour
    """
    H, W = mask.shape
    binary = (mask > 0).astype(np.uint8) * 255

    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_TC89_KCOS)
    lines = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue
        # Simplify polygon to reduce label file size
        eps     = 0.002 * cv2.arcLength(cnt, True)
        approx  = cv2.approxPolyDP(cnt, eps, True)
        pts     = approx.reshape(-1, 2).astype(np.float32)
        pts[:, 0] /= W   # normalise x
        pts[:, 1] /= H   # normalise y
        coords = ' '.join(f'{v:.6f}' for v in pts.flatten())
        lines.append(f'{class_id} {coords}')
    return lines


def get_annotations_for_sample(gt_path: Path) -> list:
    """
    Extract YOLO polygon annotations from a GT mask.
    Handles single-class (crack/hole/contamination) and
    multi-class (combined) masks.
    """
    mask  = np.array(Image.open(gt_path))
    lines = []

    # Map mask pixel values to YOLO class IDs
    VALUE_TO_CLASS = {254: 0, 253: 1, 255: 2}  # crack, hole, contamination

    for val, cls_id in VALUE_TO_CLASS.items():
        region = (mask == val).astype(np.uint8)
        if region.sum() == 0:
            continue
        lines.extend(mask_to_yolo_polygons(region, cls_id))

    return lines

# Quick test
test_gt = DATA_RAW / 'test' / 'crack' / 'gt' / '000.png'
sample_annotations = get_annotations_for_sample(test_gt)
print(f'Found {len(sample_annotations)} polygon(s) in crack/000.png')
for line in sample_annotations:
    parts = line.split()
    print(f'  class={parts[0]}, polygon points={(len(parts)-1)//2}')

In [ ]:
# ── Build the YOLO dataset ────────────────────────────────────────────────────
random.seed(42)
VAL_RATIO  = 0.20   # 20% validation

# Create directory structure
for split in ('train', 'val'):
    (YOLO_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

def collect_samples():
    """
    Gather all (rgb_path, gt_path_or_None) pairs from the test split.
    Good samples get an empty label file (background class).
    """
    samples = []
    for defect in ['crack', 'hole', 'contamination', 'combined', 'good']:
        rgb_dir = DATA_RAW / 'test' / defect / 'rgb'
        gt_dir  = DATA_RAW / 'test' / defect / 'gt'
        for rgb_path in sorted(rgb_dir.glob('*.png')):
            gt_path = gt_dir / rgb_path.name if (gt_dir.exists() and defect != 'good') else None
            samples.append({'rgb': rgb_path, 'gt': gt_path, 'defect': defect})
    return samples

all_samples = collect_samples()
random.shuffle(all_samples)
n_val   = max(1, int(len(all_samples) * VAL_RATIO))
val_set = all_samples[:n_val]
trn_set = all_samples[n_val:]

print(f'Total samples : {len(all_samples)}')
print(f'  Train       : {len(trn_set)}')
print(f'  Val         : {len(val_set)}')

In [ ]:
def write_yolo_sample(sample: dict, split: str):
    """Copy image and write YOLO label for one sample."""
    # Unique filename: defect_type + original stem
    name = f"{sample['defect']}_{sample['rgb'].stem}"

    # Copy RGB image
    dst_img = YOLO_DIR / 'images' / split / f'{name}.png'
    shutil.copy2(sample['rgb'], dst_img)

    # Write label
    dst_lbl = YOLO_DIR / 'labels' / split / f'{name}.txt'
    if sample['gt'] is not None and sample['gt'].exists():
        lines = get_annotations_for_sample(sample['gt'])
    else:
        lines = []   # good sample → empty label

    with open(dst_lbl, 'w') as f:
        f.write('\n'.join(lines))

# Write all samples
print('Writing YOLO dataset ...')
for s in tqdm(trn_set, desc='Train'):
    write_yolo_sample(s, 'train')
for s in tqdm(val_set, desc='Val  '):
    write_yolo_sample(s, 'val')

print('\nDone!')
n_trn_img = len(list((YOLO_DIR / 'images' / 'train').glob('*.png')))
n_val_img = len(list((YOLO_DIR / 'images' / 'val').glob('*.png')))
print(f'  images/train : {n_trn_img} files')
print(f'  images/val   : {n_val_img} files')

In [ ]:
# ── Verify a random label ─────────────────────────────────────────────────────
label_files = sorted((YOLO_DIR / 'labels' / 'train').glob('*.txt'))
non_empty   = [f for f in label_files if f.stat().st_size > 0]
empty       = [f for f in label_files if f.stat().st_size == 0]

print(f'Train labels  : {len(label_files)} total')
print(f'  with defects: {len(non_empty)}')
print(f'  good (empty): {len(empty)}')

# Show label distribution
class_counts = {0: 0, 1: 0, 2: 0}
for lf in non_empty:
    for line in lf.read_text().strip().split('\n'):
        if line:
            cls = int(line.split()[0])
            class_counts[cls] = class_counts.get(cls, 0) + 1

CLASS_NAMES = {0: 'crack', 1: 'hole', 2: 'contamination'}
print('\nPolygon count per class (train):')
for cls_id, cnt in class_counts.items():
    print(f'  [{cls_id}] {CLASS_NAMES[cls_id]:<15} : {cnt}')

In [ ]:
# ── Visualise YOLO labels on a random sample ──────────────────────────────────
COLORS = {0: (255, 80, 80), 1: (80, 180, 255), 2: (80, 220, 80)}

def draw_yolo_labels(img_path: Path, lbl_path: Path) -> np.ndarray:
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    overlay = img.copy()

    if lbl_path.stat().st_size == 0:
        return img

    for line in lbl_path.read_text().strip().split('\n'):
        if not line:
            continue
        parts  = line.split()
        cls_id = int(parts[0])
        coords = np.array(parts[1:], dtype=np.float32).reshape(-1, 2)
        pts    = (coords * [W, H]).astype(np.int32)
        color  = COLORS.get(cls_id, (255, 255, 0))
        cv2.fillPoly(overlay, [pts], color)
        cv2.polylines(img, [pts], True, color, 2)

    return cv2.addWeighted(img, 0.6, overlay, 0.4, 0)

# Pick samples with non-empty labels
imgs   = sorted((YOLO_DIR / 'images' / 'train').glob('*.png'))
sample_imgs = [i for i in imgs if (YOLO_DIR / 'labels' / 'train' / i.with_suffix('.txt').name).stat().st_size > 0][:4]

fig, axes = plt.subplots(1, len(sample_imgs), figsize=(5 * len(sample_imgs), 5))
for ax, img_path in zip(axes, sample_imgs):
    lbl_path = YOLO_DIR / 'labels' / 'train' / img_path.with_suffix('.txt').name
    vis = draw_yolo_labels(img_path, lbl_path)
    ax.imshow(vis)
    ax.set_title(img_path.stem, fontsize=9)
    ax.axis('off')

patches = [mpatches.Patch(color=np.array(c)/255, label=CLASS_NAMES[i]) for i, c in COLORS.items()]
plt.legend(handles=patches, loc='upper right')
plt.suptitle('YOLO Labels Verification', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(ROOT / 'experiments' / 'results' / 'yolo_labels_check.png'), dpi=120)
plt.show()

## 4. Create Dataset YAML

In [ ]:
dataset_yaml = {
    'path'  : str(YOLO_DIR),
    'train' : 'images/train',
    'val'   : 'images/val',
    'nc'    : 3,
    'names' : ['crack', 'hole', 'contamination'],
}

yaml_path = YOLO_DIR / 'dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False, sort_keys=False)

print('dataset.yaml content:')
print('-' * 40)
print(yaml_path.read_text())

## 5. Train YOLOv8-Seg

| Model | Params | Speed | Recommended for |
|---|---|---|---|
| `yolov8n-seg` | 3.4M | fastest | Small datasets like ours |
| `yolov8s-seg` | 11.8M | fast | Medium datasets |
| `yolov8m-seg` | 27.3M | moderate | Large datasets |

In [ ]:
from ultralytics import YOLO
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Load pretrained model ─────────────────────────────────────────────────────
# yolov8n-seg : nano   — best for our small dataset (110 samples)
# yolov8s-seg : small  — slightly better accuracy, slower
MODEL_NAME = 'yolov8n-seg'
model = YOLO(f'{MODEL_NAME}.pt')   # downloads pretrained weights automatically
print(f'Model loaded : {MODEL_NAME}')
print(f'Parameters   : {sum(p.numel() for p in model.model.parameters()) / 1e6:.1f} M')

In [ ]:
# ── Training configuration ────────────────────────────────────────────────────
# FIXES applied for tiny defects (0.23% - 1.32% of image area):
#   1. imgsz=800  → preserve original resolution (800×800 images, no downscale)
#   2. patience=25 → give more time to learn tiny features
#   3. overlap_mask=True → better mask handling for overlapping regions

TRAIN_CFG = dict(
    data        = str(yaml_path),
    epochs      = 150,
    imgsz       = 800,          # FIX 1: original image size (was 640, losing tiny defects)
    batch       = 4,            # reduced from 8 → fits memory at 800px
    device      = device,
    workers     = 0,
    project     = str(ROOT / 'experiments' / 'yolo_runs'),
    name        = f'{MODEL_NAME}_bagel',
    exist_ok    = True,

    # Optimizer
    optimizer   = 'AdamW',
    lr0         = 0.001,
    lrf         = 0.01,
    weight_decay= 0.0005,
    warmup_epochs = 5,

    # Augmentation
    hsv_h       = 0.015,
    hsv_s       = 0.5,
    hsv_v       = 0.4,
    degrees     = 180,
    translate   = 0.1,
    scale       = 0.3,
    flipud      = 0.5,
    fliplr      = 0.5,
    mosaic      = 0.5,
    copy_paste  = 0.5,          # increased: helps with tiny defect instances
    erasing     = 0.2,

    # Loss weights — boost box/seg for tiny objects
    box         = 10.0,         # increased from 7.5 → penalise missed tiny boxes more
    cls         = 0.5,
    dfl         = 1.5,

    # Early stopping — FIX 2: give more time to learn tiny features
    patience    = 25,           # was 10 → too aggressive for 0.23% defects

    # Mask settings
    overlap_mask = True,        # FIX 3: better for overlapping defect regions

    # Logging
    plots       = True,
    save        = True,
    save_period = 10,
    verbose     = True,
)

print('Training configuration (fixes for tiny defects):')
print(f"  {'imgsz':<22} = {TRAIN_CFG['imgsz']}  ← was 640, now 800 (original res)")
print(f"  {'batch':<22} = {TRAIN_CFG['batch']}   ← reduced to fit 800px in memory")
print(f"  {'patience (ES)':<22} = {TRAIN_CFG['patience']}  ← was 10, now 25")
print(f"  {'box loss weight':<22} = {TRAIN_CFG['box']}  ← was 7.5, increased for tiny objects")
print(f"  {'copy_paste':<22} = {TRAIN_CFG['copy_paste']}  ← increased for tiny defect augmentation")
print(f"  {'overlap_mask':<22} = {TRAIN_CFG['overlap_mask']}")
print()
print('Defect sizes at 800x800:')
print('  contamination  ~1490 px²  (0.23%)  ← very small')
print('  hole           ~1535 px²  (0.24%)  ← very small')
print('  crack          ~5268 px²  (0.82%)  ← small')
print('  combined       ~8433 px²  (1.32%)  ← small')

In [ ]:
# ── Launch training ───────────────────────────────────────────────────────────
print('Starting training ...')
results = model.train(**TRAIN_CFG)
print('\nTraining complete!')

## 6. Results & Evaluation

In [ ]:
# ── Load best checkpoint ──────────────────────────────────────────────────────
run_dir   = ROOT / 'experiments' / 'yolo_runs' / f'{MODEL_NAME}_bagel'
best_ckpt = run_dir / 'weights' / 'best.pt'

print(f'Best checkpoint : {best_ckpt}')
best_model = YOLO(str(best_ckpt))

# ── Validation metrics ────────────────────────────────────────────────────────
# conf=0.1 → lower threshold for tiny defects (default 0.25 filters too many)
# iou=0.3  → relaxed IoU for small objects (default 0.6 is strict)
metrics = best_model.val(
    data    = str(yaml_path),
    imgsz   = 800,
    conf    = 0.1,              # FIX: lower conf for tiny defects
    iou     = 0.3,              # FIX: relaxed IoU threshold
    device  = device,
    verbose = True,
)

print('\n' + '='*50)
print('VALIDATION METRICS')
print('='*50)
print(f'  mAP50 (box)       : {metrics.box.map50:.4f}')
print(f'  mAP50-95 (box)    : {metrics.box.map:.4f}')
print(f'  mAP50 (mask)      : {metrics.seg.map50:.4f}')
print(f'  mAP50-95 (mask)   : {metrics.seg.map:.4f}')
print(f'  Precision         : {metrics.box.mp:.4f}')
print(f'  Recall            : {metrics.box.mr:.4f}')

In [ ]:
# ── Per-class metrics ─────────────────────────────────────────────────────────
print('\nPer-class AP50 (mask):')
class_names = ['crack', 'hole', 'contamination']
for i, (name, ap) in enumerate(zip(class_names, metrics.seg.ap50)):
    print(f'  [{i}] {name:<18} : {ap:.4f}')

# ── Training curves ───────────────────────────────────────────────────────────
results_csv = run_dir / 'results.csv'
if results_csv.exists():
    import pandas as pd
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    plot_pairs = [
        ('train/box_loss', 'val/box_loss',  'Box Loss',       axes[0, 0]),
        ('train/seg_loss', 'val/seg_loss',  'Seg Loss',       axes[0, 1]),
        ('train/cls_loss', 'val/cls_loss',  'Cls Loss',       axes[0, 2]),
        ('metrics/mAP50(M)', None,          'mAP50 Mask',     axes[1, 0]),
        ('metrics/mAP50-95(M)', None,       'mAP50-95 Mask',  axes[1, 1]),
        ('metrics/precision(M)', 'metrics/recall(M)', 'Precision/Recall', axes[1, 2]),
    ]
    for col1, col2, title, ax in plot_pairs:
        if col1 in df.columns:
            ax.plot(df[col1], label='train' if col2 else col1.split('/')[-1], linewidth=2)
        if col2 and col2 in df.columns:
            ax.plot(df[col2], label='val' if col1.startswith('train') else col2.split('/')[-1],
                    linestyle='--', linewidth=2)
        ax.set_title(title, fontsize=12)
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.suptitle(f'YOLOv8-Seg Training — {MODEL_NAME}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(ROOT / 'experiments' / 'results' / 'yolo_training_curves.png'), dpi=120)
    plt.show()
    print('Saved → experiments/results/yolo_training_curves.png')

## 7. Inference Demo

In [ ]:
# ── Run inference on validation images ───────────────────────────────────────
CONF_THRESHOLD = 0.25
IOU_THRESHOLD  = 0.45

val_images = sorted((YOLO_DIR / 'images' / 'val').glob('*.png'))
# Show 6 examples that have defects
defect_imgs = [i for i in val_images if not i.stem.startswith('good')][:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
COLORS_BGR = {0: (80, 80, 255), 1: (255, 180, 80), 2: (80, 220, 80)}

for ax, img_path in zip(axes, defect_imgs):
    preds = best_model.predict(
        source    = str(img_path),
        conf      = CONF_THRESHOLD,
        iou       = IOU_THRESHOLD,
        imgsz     = 640,
        device    = device,
        verbose   = False,
    )[0]

    img    = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    overlay = img_rgb.copy()
    H, W   = img_rgb.shape[:2]
    detected = []

    if preds.masks is not None:
        masks  = preds.masks.data.cpu().numpy()
        boxes  = preds.boxes
        for j, (mask, box) in enumerate(zip(masks, boxes)):
            cls_id = int(box.cls.item())
            conf   = float(box.conf.item())
            mask_r = cv2.resize(mask, (W, H))
            color  = np.array(COLORS_BGR.get(cls_id, (200, 200, 200)), dtype=np.uint8)
            overlay[mask_r > 0.5] = (overlay[mask_r > 0.5] * 0.4 + color * 0.6).astype(np.uint8)
            detected.append(f'{CLASS_NAMES[cls_id]} {conf:.2f}')

    ax.imshow(overlay)
    status = ', '.join(detected) if detected else 'No defect detected'
    ax.set_title(f'{img_path.stem}\n{status}', fontsize=9)
    ax.axis('off')

patches = [mpatches.Patch(color=np.array(list(c)[::-1])/255, label=CLASS_NAMES[i])
           for i, c in COLORS_BGR.items()]
fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=11)
plt.suptitle('YOLOv8-Seg — Inference on Validation Set', fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig(str(ROOT / 'experiments' / 'results' / 'yolo_inference_demo.png'), dpi=120)
plt.show()
print('Saved → experiments/results/yolo_inference_demo.png')

In [ ]:
# ── Confusion matrix & F1 curve ───────────────────────────────────────────────
conf_img = run_dir / 'confusion_matrix_normalized.png'
f1_img   = run_dir / 'MaskF1_curve.png'

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, img_p, title in [
    (axes[0], conf_img, 'Confusion Matrix (normalised)'),
    (axes[1], f1_img,   'F1-Confidence Curve'),
]:
    if img_p.exists():
        ax.imshow(plt.imread(str(img_p)))
        ax.set_title(title, fontsize=12)
    else:
        ax.text(0.5, 0.5, f'{img_p.name} not found', ha='center', va='center')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 8. Summary

In [ ]:
print('=' * 55)
print('  YOLO SEGMENTATION — TRAINING SUMMARY')
print('=' * 55)
print(f'  Model          : {MODEL_NAME}')
print(f'  Dataset        : MVTec 3D-AD — bagel')
print(f'  Classes        : crack / hole / contamination')
print(f'  Train samples  : {len(trn_set)}')
print(f'  Val samples    : {len(val_set)}')
print(f'  Image size     : 640 × 640')
print(f'  Device         : {device}')
print()
print(f'  mAP50  (box)   : {metrics.box.map50:.4f}')
print(f'  mAP50  (mask)  : {metrics.seg.map50:.4f}')
print(f'  mAP50-95 (mask): {metrics.seg.map:.4f}')
print(f'  Precision      : {metrics.box.mp:.4f}')
print(f'  Recall         : {metrics.box.mr:.4f}')
print()
print(f'  Best checkpoint: {best_ckpt}')
print('=' * 55)